In [1]:
#| default_exp rest_contrastive

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:
#| export
from rest.core import init_instance, process_seq
singleton, model_path = init_instance()

In [4]:
model_path = 'pelevin'

In [5]:
#| export
seq_length = 1024

model_path = f'./models/large/{model_path}'
from transformers import GPT2LMHeadModel,GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path).half()
model.cuda()
model.eval();

In [32]:
sum(p.numel() for p in model.parameters())

774030080

In [36]:
student_tokenizer = GPT2Tokenizer.from_pretrained('sberbank-ai/rugpt3small_based_on_gpt2')

In [27]:
student_lm = GPT2LMHeadModel.from_pretrained('sberbank-ai/rugpt3small_based_on_gpt2').half().cuda()

In [ ]:
sum(p.numel() for p in student_lm.parameters())

In [76]:
from tokenizers.decoders import ByteLevel
decoder = ByteLevel()

In [77]:
def uncode(tokenizer):
    keys = tokenizer.get_vocab().keys()
    keys = [decoder.decode(k) for k in keys]
    return set(keys)

In [78]:
tbig = uncode(tokenizer)
tsmall = uncode(student_tokenizer)

In [79]:
len(list(tbig-tsmall))

15042

In [80]:
len(list(tsmall-tbig))

15042

In [81]:
len(list(tsmall&tbig))

35068

In [84]:
tokenizer.encode('слышал')

[9478, 293]

In [85]:
student_tokenizer.encode('слышал')

[41405]

In [28]:
def ignore_prefix_prepare_inputs_for_generation(input_ids, past=None, **kwargs):
            
    token_type_ids = kwargs.get("token_type_ids", None)
    # only last token for inputs_ids if past is defined in kwargs
    input_ids = input_ids[:, -1].unsqueeze(-1)
    if token_type_ids is not None:
        token_type_ids = token_type_ids[:, -1].unsqueeze(-1)

    attention_mask = kwargs.get("attention_mask", None)
    position_ids = kwargs.get("position_ids", None)

    if attention_mask is not None and position_ids is None:
        # create position_ids on the fly for batch generation
        position_ids = attention_mask.long().cumsum(-1) - 1
        position_ids.masked_fill_(attention_mask == 0, 1)
        position_ids = position_ids[:, -1].unsqueeze(-1)
    else:
        position_ids = None

    return {
        "input_ids": input_ids,
        "past_key_values": past,
        "use_cache": kwargs.get("use_cache"),
        "position_ids": position_ids,
        "attention_mask": attention_mask,
        "token_type_ids": token_type_ids,
    }

In [29]:
student_lm.prepare_inputs_for_generation = ignore_prefix_prepare_inputs_for_generation

In [30]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    encoded_prompt = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt").cuda()
    encoded_prompt = encoded_prompt[:,length-(seq_length-1):]
    bad_words_ids = [tokenizer.encode('[')[0], tokenizer.encode('(')[0], tokenizer.encode('1\xa01')[1]]
    linebreak = tokenizer.encode("1\n1")[1]
    lb2 = tokenizer.encode("1 \n")[1]
    bad_words_ids += [] if allow_linebreak else [linebreak, lb2]
    bad_words_ids = [[b] for b in bad_words_ids] + [[linebreak,linebreak]]
    output_sequences = model.generate(
            input_ids=encoded_prompt,
            max_length=length + len(encoded_prompt[0]),
            temperature=1,
            top_k=0,
            top_p=0.9,
            do_sample=True,num_return_sequences=num_samples,
            bad_words_ids = bad_words_ids,
            student_lm=student_lm,
            teacher_student=True,
            model_kwargs_student={},
            st_coef=1.0,
        )
    
    if len(output_sequences.shape) > 2:
            output_sequences.squeeze_()
    generated_sequences = []
    for generated_sequence_idx, generated_sequence in enumerate(output_sequences):
        generated_sequence = generated_sequence.tolist()
        text = tokenizer.decode(generated_sequence, clean_up_tokenization_spaces=True)
        total_sequence = text[len(tokenizer.decode(encoded_prompt[0], clean_up_tokenization_spaces=True)) :]
        generated_sequences.append(total_sequence)

    return process_seq(generated_sequences)

In [31]:
%%time
get_sample(' - ты кто?', 50, 4, False)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


setting the adaptive thresholding
[]


RuntimeError: The size of tensor a (50257) must match the size of tensor b (50264) at non-singleton dimension 1